In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("power_distance.csv")

# Keep only the required columns
columns_to_keep = ["worker_id", "output_id", "annotation_range"]
df_filtered = df[columns_to_keep]

# Save the new file
df_filtered.to_csv("power_distance_prepared.csv", index=False)

print("✅ New file saved as 'power_distance_prepared.csv'")


✅ New file saved as 'power_distance_prepared.csv'


In [ ]:
import numpy as np

# ---- Config ----
CSV = "power_distance_prepared.csv"
BOOTSTRAP_SAMPLES = 1000  # increase to 5000+ for tighter CI if needed
RNG_SEED = 42

# ---- Load & pivot ----
df = pd.read_csv(CSV)  # expects: worker_id, output_id, annotation_range
mat = df.pivot_table(index="output_id",
                     columns="worker_id",
                     values="annotation_range",
                     aggfunc="mean")

# ---- Fill missing with row (item) means so all items contribute ----
X = mat.values.astype(float)
row_means = np.nanmean(X, axis=1, keepdims=True)
# If a row is entirely NaN, its mean is NaN; replace with overall mean to avoid NaNs.
overall_mean = np.nanmean(X)
row_means = np.where(np.isnan(row_means), overall_mean, row_means)
nan_rows, nan_cols = np.where(np.isnan(X))
X[nan_rows, nan_cols] = row_means[nan_rows, 0]

n, k = X.shape  # n items, k raters

def icc2_abs_agreement(x: np.ndarray) -> float:
    """Shrout & Fleiss ICC(2,1): two-way random, absolute agreement."""
    n_, k_ = x.shape
    gm = x.mean()
    row_means = x.mean(axis=1, keepdims=True)     # items
    col_means = x.mean(axis=0, keepdims=True)     # raters

    SSR = k_ * np.sum((row_means - gm) ** 2)                       # between items
    SSC = n_ * np.sum((col_means - gm) ** 2)                       # between raters
    SSE = np.sum((x - row_means - col_means + gm) ** 2)            # residual

    MSR = SSR / (n_ - 1)
    MSC = SSC / (k_ - 1)
    MSE = SSE / ((n_ - 1) * (k_ - 1))

    denom = MSR + (k_ - 1) * MSE + (k_ * (MSC - MSE) / n_)
    return float((MSR - MSE) / denom) if denom != 0 else np.nan

# ---- Point estimate ----
icc_point = icc2_abs_agreement(X)

# ---- Bootstrap 95% CI (resample items with replacement) ----
rng = np.random.default_rng(RNG_SEED)
idx = np.arange(n)
samples = []
for _ in range(BOOTSTRAP_SAMPLES):
    sel = rng.choice(idx, size=n, replace=True)
    x_b = X[sel, :]
    samples.append(icc2_abs_agreement(x_b))
samples = np.array(samples, dtype=float)
ci_low, ci_high = np.nanpercentile(samples, [2.5, 97.5])

# ---- Interpretation ----
if np.isnan(icc_point):
    interp = "insufficient data"
elif icc_point > 0.7:
    interp = "High agreement (strong)"
elif icc_point >= 0.4:
    interp = "Moderate agreement"
else:
    interp = "Low agreement / subjective stream"

# ---- Report ----
print("----- ICC(2,1) (Two-way random, absolute-agreement) -----")
print(f"Items (n): {n}, Raters (k): {k}")
print(f"ICC(2,1): {icc_point:.3f}")
print(f"95% CI  : [{ci_low:.3f}, {ci_high:.3f}]  (bootstrap, {BOOTSTRAP_SAMPLES} resamples)")
print(f"Interpretation: {interp}")


----- ICC(2,1) (Two-way random, absolute-agreement) -----
Items (n): 270, Raters (k): 6
ICC(2,1): 0.959
95% CI  : [0.946, 0.971]  (bootstrap, 1000 resamples)
Interpretation: High agreement (strong)


In [ ]:
# === Load cleaned dataset ===
df = pd.read_csv("power_distance_prepared.csv")

# Average duplicates (worker_id + output_id) ===
df = df.groupby(["worker_id", "output_id"], as_index=False)["annotation_range"].mean()

# Pivot automatically ===
mat = df.pivot_table(index="output_id",
                     columns="worker_id",
                     values="annotation_range",
                     aggfunc="mean")

# Handle missing values ===
# Leave missing as NaN for math ops, but when saving, we’ll display “—”
mat_display = mat.copy().where(~mat.isna(), other="—")

# Remove rows with only one rating (for correlation calc) ===
mat_for_corr = mat.loc[mat.count(axis=1) > 1]

# Compute pairwise Pearson correlation matrix ===
corr_matrix = mat_for_corr.corr(method="pearson", min_periods=2)

# Save outputs ===
mat_display.to_csv("power_distance_table_270x7.csv")
corr_matrix.to_csv("power_distance_pairwise_corr.csv")

# Print summary ===
print("✅ Saved full 270x7 table: power_distance_table_270x7.csv")
print("✅ Saved pairwise correlation matrix: power_distance_pairwise_corr.csv")
print("\nPairwise Correlation Matrix:")
print(corr_matrix.round(3))

✅ Saved full 270x7 table: power_distance_table_270x7.csv
✅ Saved pairwise correlation matrix: power_distance_pairwise_corr.csv

Pairwise Correlation Matrix:
worker_id       A133HLDA3JJV0M  A17QWRCEPG0775  A1E80Q7U5UXLMS  \
worker_id                                                        
A133HLDA3JJV0M           1.000           0.735           0.830   
A17QWRCEPG0775           0.735           1.000           0.932   
A1E80Q7U5UXLMS           0.830           0.932           1.000   
A2P3XETFFG5K7J           0.908             NaN           0.806   
A3240Z3SG99X06           0.806             NaN           0.822   
A39KQ6Q83RH3NO           0.666           0.430             NaN   

worker_id       A2P3XETFFG5K7J  A3240Z3SG99X06  A39KQ6Q83RH3NO  
worker_id                                                       
A133HLDA3JJV0M           0.908           0.806           0.666  
A17QWRCEPG0775             NaN             NaN           0.430  
A1E80Q7U5UXLMS           0.806           0.822        

In [ ]:
# === Load cleaned dataset ===
df = pd.read_csv("power_distance_prepared.csv")

# Average duplicates ===
df = df.groupby(["worker_id", "output_id"], as_index=False)["annotation_range"].mean()

# Pivot to items × raters matrix ===
mat = df.pivot_table(index="output_id",
                     columns="worker_id",
                     values="annotation_range",
                     aggfunc="mean")

# Compute Spearman and Kendall correlation matrices ===
spearman_corr = mat.corr(method="spearman", min_periods=2)
kendall_corr  = mat.corr(method="kendall",  min_periods=2)

# Save both results ===
spearman_corr.to_csv("power_distance_spearman_corr.csv")
kendall_corr.to_csv("power_distance_kendall_corr.csv")

# Display summary ===
print("✅ Saved Spearman rank correlation matrix: power_distance_spearman_corr.csv")
print("✅ Saved Kendall rank correlation matrix: power_distance_kendall_corr.csv")

print("\nSpearman Rank Correlation Matrix:")
print(spearman_corr.round(3))

print("\nKendall Rank Correlation Matrix:")
print(kendall_corr.round(3))


✅ Saved Spearman rank correlation matrix: power_distance_spearman_corr.csv
✅ Saved Kendall rank correlation matrix: power_distance_kendall_corr.csv

Spearman Rank Correlation Matrix:
worker_id       A133HLDA3JJV0M  A17QWRCEPG0775  A1E80Q7U5UXLMS  \
worker_id                                                        
A133HLDA3JJV0M           1.000           0.745           0.786   
A17QWRCEPG0775           0.745           1.000           0.895   
A1E80Q7U5UXLMS           0.786           0.895           1.000   
A2P3XETFFG5K7J           0.914             NaN           0.809   
A3240Z3SG99X06           0.777             NaN           0.782   
A39KQ6Q83RH3NO           0.612           0.323             NaN   

worker_id       A2P3XETFFG5K7J  A3240Z3SG99X06  A39KQ6Q83RH3NO  
worker_id                                                       
A133HLDA3JJV0M           0.914           0.777           0.612  
A17QWRCEPG0775             NaN             NaN           0.323  
A1E80Q7U5UXLMS           0.8